# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 5 — Bronze: limpar sem inventar

## 🎯 Objetivo

Construir a camada Bronze: remover duplicatas, corrigir tipos e descartar linhas impossíveis — sem tocar em lógica de negócio ainda.


**Importante:** **Rota B — DuckDB + Python/Google Colab**



## Configuração Inicial

Arquivos esperados:
- `customers_synthetic.csv` — 9.993 registros
- `transactions_synthetic.csv` — 100.000 registros


In [25]:
# Instalação de dependências para o ambiente DuckDB
!pip -q install duckdb

In [26]:
# Configuração do ambiente DuckDB criação de diretórios para dados
import os
import glob
import shutil
import duckdb
import pandas as pd

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)
os.makedirs("bigdata/bronze", exist_ok=True)

con = duckdb.connect()
print("Estrutura criada.")

Estrutura criada.


## Upload dos arquivos de entrada

In [27]:
# Upload de arquivos CSV para o ambiente Colab
# from google.colab import files

# uploaded = files.upload()

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")




✓ Os 3 datasets foram encontrados.


In [28]:
# Distribuição dos arquivos CSV em diretórios raw específicos para processamento
for name in uploaded:
    # Copia o arquivo 'customers_synthetic.csv' para o diretório de clientes raw
    if name == "../customers_synthetic.csv":
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    # Copia o arquivo 'transactions_synthetic.csv' para o diretório de transações raw
    elif name == "../transactions_synthetic.csv":
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw:")
# Lista todos os arquivos CSV presentes nos diretórios da camada raw
for path in glob.glob("bigdata/raw/*/*.csv"):
    print("-", path)

Arquivos Raw:
- bigdata/raw\customers\customers_synthetic.csv
- bigdata/raw\transactions\transactions_synthetic.csv


## Validação da camada Raw

In [29]:
# Contagem de registros e exibição de amostras da camada Raw para verificação inicial
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

# Exibe a contagem total de registros no arquivo raw de clientes
print("Customers Raw:", con.sql(
    f"SELECT COUNT(*) FROM read_csv_auto('{customers_raw}')"
).fetchone()[0])

# Exibe a contagem total de registros no arquivo raw de transações
print("Transactions Raw:", con.sql(
    f"SELECT COUNT(*) FROM read_csv_auto('{transactions_raw}')"
).fetchone()[0])

print("\nAmostra de customers:")
# Mostra as primeiras 5 linhas do arquivo raw de clientes
con.sql(f"SELECT * FROM read_csv_auto('{customers_raw}') LIMIT 5").show()

print("\nAmostra de transactions:")
# Mostra as primeiras 5 linhas do arquivo raw de transações
con.sql(f"SELECT * FROM read_csv_auto('{transactions_raw}') LIMIT 5").show()

Customers Raw: 9993
Transactions Raw: 100000

Amostra de customers:
┌─────────────┬─────────────────────┬────────────────┬──────────────────────────┬───────────┬──────────────┬────────────┐
│ customer_id │        name         │      cpf       │          email           │  segment  │ credit_score │ created_at │
│    int64    │       varchar       │    varchar     │         varchar          │  varchar  │    int64     │    date    │
├─────────────┼─────────────────────┼────────────────┼──────────────────────────┼───────────┼──────────────┼────────────┤
│           1 │ Ana Laura Campos    │ 943.065.218-42 │ igor46@example.com       │ Premium   │          426 │ 2026-07-03 │
│           2 │ Mariah Caldeira     │ 586.237.094-38 │ jose48@example.com       │ High-Risk │          481 │ 2026-06-17 │
│           3 │ Kevin Cavalcante    │ 530.629.814-15 │ theoda-costa@example.org │ Standard  │          708 │ 2025-11-06 │
│           4 │ Maria Laura Freitas │ 725.130.864-90 │ castrolucas@example.org

## Passo 1 — Bronze de clientes

Regras:
- `DISTINCT`;
- `customer_id IS NOT NULL`;
- `credit_score BETWEEN 300 AND 900`;
- `credit_score` → `INT`;
- `created_at` → `DATE`.

Nenhuma regra de negócio adicional será aplicada.


In [30]:
# Criação da tabela bronze_customers remoção de duplicados correção de tipos e filtragem inicial
con.sql(f'''
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id,
    name,
    cpf,
    email,
    segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
''')

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘



In [31]:
# Exibição de amostras e esquema de tipos da tabela bronze_customers para validação
print("Amostra:")
# Mostra as primeiras 10 linhas da tabela bronze_customers para inspeção visual
con.sql("SELECT * FROM bronze_customers LIMIT 10").show()

print("Tipos:")
# Descreve a estrutura e os tipos de dados das colunas em bronze_customers
con.sql("DESCRIBE bronze_customers").show()

Amostra:
┌─────────────┬──────────────────────────┬────────────────┬───────────────────────────┬───────────┬──────────────┬────────────┐
│ customer_id │           name           │      cpf       │           email           │  segment  │ credit_score │ created_at │
│    int64    │         varchar          │    varchar     │          varchar          │  varchar  │    int32     │    date    │
├─────────────┼──────────────────────────┼────────────────┼───────────────────────────┼───────────┼──────────────┼────────────┤
│           1 │ Ana Laura Campos         │ 943.065.218-42 │ igor46@example.com        │ Premium   │          426 │ 2026-07-03 │
│           2 │ Mariah Caldeira          │ 586.237.094-38 │ jose48@example.com        │ High-Risk │          481 │ 2026-06-17 │
│           3 │ Kevin Cavalcante         │ 530.629.814-15 │ theoda-costa@example.org  │ Standard  │          708 │ 2025-11-06 │
│           5 │ Srta. Mirella Moura      │ 570.814.239-14 │ emilly20@example.com      │ Premium

## Passo 2 — Bronze de transações

Regras:
- `DISTINCT`;
- `amount > 0`;
- `customer_id IS NOT NULL`;
- `amount` → `FLOAT`;
- `risk_score` → `FLOAT`;
- `timestamp` → `TIMESTAMP`;
- `is_fraud` convertido explicitamente com `CASE WHEN`.



In [32]:
# Criação da tabela bronze_transactions remoção de duplicados correção de tipos e filtragem de valores
con.sql(f'''
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id,
    customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type,
    status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE
        WHEN is_fraud = 'True' THEN true
        ELSE false
    END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
''')

con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [33]:
# Exibição de amostras e esquema de tipos da tabela bronze_transactions
print("Amostra:")
con.sql("SELECT * FROM bronze_transactions LIMIT 10").show()

print("Tipos:")
con.sql("DESCRIBE bronze_transactions").show()

Amostra:
┌────────────────┬─────────────┬───────────┬──────────────────┬──────────┬────────────┬──────────┬─────────────────────┐
│ transaction_id │ customer_id │  amount   │ transaction_type │  status  │ risk_score │ is_fraud │         ts          │
│     int64      │    int64    │   float   │     varchar      │ varchar  │   float    │ boolean  │      timestamp      │
├────────────────┼─────────────┼───────────┼──────────────────┼──────────┼────────────┼──────────┼─────────────────────┤
│          22956 │        3412 │ 442.66748 │ pagamento        │ approved │   77.97588 │ false    │ 2024-12-23 00:00:00 │
│          84496 │        5793 │ 51.343494 │ compra           │ approved │  87.036026 │ false    │ 2024-12-23 00:00:00 │
│          21474 │        9554 │ 25.128494 │ saque            │ declined │  13.112448 │ false    │ 2024-12-23 00:00:00 │
│          55985 │        4236 │  533.3962 │ pagamento        │ approved │   68.48584 │ false    │ 2024-12-23 00:00:00 │
│          88846 │     

## Validações da Bronze

In [34]:
# Análise da faixa de valores de transações e distribuição de fraudes
print("Faixa de valores:")
con.sql('''
SELECT COUNT(*) AS total,
       MIN(amount) AS min_amount,
       MAX(amount) AS max_amount
FROM bronze_transactions
''').show()

print("Distribuição de fraude:")
con.sql('''
SELECT is_fraud,
       COUNT(*) AS quantidade,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual
FROM bronze_transactions
GROUP BY is_fraud
ORDER BY is_fraud
''').show()

Faixa de valores:
┌────────┬────────────┬────────────┐
│ total  │ min_amount │ max_amount │
│ int64  │   float    │   float    │
├────────┼────────────┼────────────┤
│ 100000 │       10.0 │  22927.023 │
└────────┴────────────┴────────────┘

Distribuição de fraude:
┌──────────┬────────────┬────────────┐
│ is_fraud │ quantidade │ percentual │
│ boolean  │   int64    │   double   │
├──────────┼────────────┼────────────┤
│ false    │      98167 │      98.17 │
│ true     │       1833 │       1.83 │
└──────────┴────────────┴────────────┘



In [35]:
# Contagem de transações com valores inválidos e customer_id nulo
print("amount <= 0:")
con.sql('''
SELECT COUNT(*) AS quantidade
FROM bronze_transactions
WHERE amount <= 0
''').show()

print("customer_id nulo:")
con.sql('''
SELECT COUNT(*) AS quantidade
FROM bronze_transactions
WHERE customer_id IS NULL
''').show()

amount <= 0:
┌────────────┐
│ quantidade │
│   int64    │
├────────────┤
│          0 │
└────────────┘

customer_id nulo:
┌────────────┐
│ quantidade │
│   int64    │
├────────────┤
│          0 │
└────────────┘



## Passo 3 — Checar duplicatas

In [36]:
# Verificação de transaction_id duplicados na camada Bronze
duplicados = con.sql('''
SELECT transaction_id, COUNT(*) AS c
FROM bronze_transactions
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY c DESC
''').df()

print(f"Quantidade de transaction_id duplicados: {len(duplicados)}")

if len(duplicados) == 0:
    print("OK — nenhum transaction_id duplicado.")
else:
    display(duplicados.head(20))

Quantidade de transaction_id duplicados: 0
OK — nenhum transaction_id duplicado.



## Comparação Raw × Bronze

A redução de registros é esperada quando duplicidades ou registros estruturalmente inválidos são encontrados.


In [37]:
# Comparação da contagem de registros entre as camadas Raw e Bronze
# Obtém a contagem de registros brutos de clientes diretamente do CSV
raw_customers_count = con.sql(
    f"SELECT COUNT(*) FROM read_csv_auto('{customers_raw}')"
).fetchone()[0]

# Obtém a contagem de registros de clientes após o processamento Bronze
bronze_customers_count = con.sql(
    "SELECT COUNT(*) FROM bronze_customers"
).fetchone()[0]

# Obtém a contagem de registros brutos de transações diretamente do CSV
raw_transactions_count = con.sql(
    f"SELECT COUNT(*) FROM read_csv_auto('{transactions_raw}')"
).fetchone()[0]

# Obtém a contagem de registros de transações após o processamento Bronze
bronze_transactions_count = con.sql(
    "SELECT COUNT(*) FROM bronze_transactions"
).fetchone()[0]

# Cria um DataFrame Pandas para comparar as contagens de registros
comparacao = pd.DataFrame({
    "camada": ["customers", "transactions"],
    "raw": [raw_customers_count, raw_transactions_count],
    "bronze": [bronze_customers_count, bronze_transactions_count],
})

# Calcula o número de registros removidos entre as camadas raw e bronze
comparacao["removidos"] = comparacao["raw"] - comparacao["bronze"]
# Calcula o percentual de registros removidos, arredondando para duas casas decimais
comparacao["percentual_removido"] = (
    comparacao["removidos"] / comparacao["raw"] * 100
).round(2)

# Exibe o DataFrame de comparação resultante
display(comparacao)

,camada,raw,bronze,removidos,percentual_removido
0,customers,9993,9993,0,0.0
1,transactions,100000,100000,0,0.0


## Passo 4 — Persistir a camada Bronze em disco (Parquet)

In [38]:
# Salvamento das tabelas bronze_customers e bronze_transactions em formato Parquet
con.sql('''
COPY bronze_customers
TO 'bigdata/bronze/customers.parquet'
(FORMAT PARQUET)
''')

con.sql('''
COPY bronze_transactions
TO 'bigdata/bronze/transactions.parquet'
(FORMAT PARQUET)
''')

print("Bronze layer salva em bigdata/bronze/")

Bronze layer salva em bigdata/bronze/


In [39]:
# Listagem dos arquivos Parquet gerados na camada Bronze
for path in glob.glob("bigdata/bronze/*"):
    print("-", path)

- bigdata/bronze\customers.parquet
- bigdata/bronze\transactions.parquet


## Validação dos arquivos Parquet

In [40]:
# Verificação da contagem de registros e tipos nos arquivos Parquet da Bronze
print("Customers Parquet:")
con.sql('''
SELECT COUNT(*) AS total
FROM read_parquet('bigdata/bronze/customers.parquet')
''').show()

print("Transactions Parquet:")
con.sql('''
SELECT COUNT(*) AS total
FROM read_parquet('bigdata/bronze/transactions.parquet')
''').show()

print("Tipos de transactions:")
con.sql('''
DESCRIBE SELECT *
FROM read_parquet('bigdata/bronze/transactions.parquet')
''').show()

Customers Parquet:
┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

Transactions Parquet:
┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘

Tipos de transactions:
┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ transaction_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ amount           │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ transaction_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ status           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ risk_score       │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ is_fraud         │ BOOLEAN     │ YES     │ NULL    │ NUL

## ✅ Checkpoint

- `bronze_customers` criado;
- `bronze_transactions` criado;
- zero `transaction_id` duplicado;
- todos os `amount` maiores que zero;
- fraude próxima de 1,83%.


In [41]:
# Verificação dos critérios de sucesso para o checkpoint do Lab 5
checks = {}

checks["bronze_customers_criada"] = bronze_customers_count > 0
checks["bronze_transactions_criada"] = bronze_transactions_count > 0
checks["zero_transaction_id_duplicado"] = len(duplicados) == 0

invalid_amounts = con.sql('''
SELECT COUNT(*) FROM bronze_transactions WHERE amount <= 0
''').fetchone()[0]

checks["amount_maior_que_zero"] = invalid_amounts == 0

fraud_count = con.sql('''
SELECT COUNT(*) FROM bronze_transactions WHERE is_fraud = true
''').fetchone()[0]

fraud_pct = fraud_count / bronze_transactions_count * 100
checks["fraude_proxima_de_1_83_porcento"] = 1.0 <= fraud_pct <= 3.0

checkpoint = pd.DataFrame([
    {"item": item, "resultado": "OK" if ok else "REVISAR"}
    for item, ok in checks.items()
])

display(checkpoint)

print(f"Fraudes na Bronze: {fraud_count}")
print(f"Percentual de fraude: {fraud_pct:.2f}%")

,item,resultado
0,bronze_customers_criada,OK
1,bronze_transactions_criada,OK
2,zero_transaction_id_duplicado,OK
3,amount_maior_que_zero,OK
4,fraude_proxima_de_1_83_porcento,OK


Fraudes na Bronze: 1833
Percentual de fraude: 1.83%
